In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from timeit import default_timer as timer
from pathlib import Path
import json
import csv

In [32]:
from matplotlib import rcParams
import matplotlib.patches as mpatches

def compare_runs(run_names, run_labels=None, source_dir="results", save_dir="comparison_graphs", 
                title_fontsize=30, axis_fontsize=25, tick_fontsize=23, legend_fontsize=25, 
                figure_size=(14, 14), dpi=300, line_styles=None, line_colors=None, line_width=3):
    # Create directory for comparison graphs if it doesn't exist
    os.makedirs(save_dir, exist_ok=True)
    
    # Set global font properties
    plt.rcParams.update({
        'font.size': tick_fontsize,
        'axes.titlesize': title_fontsize,
        'axes.labelsize': axis_fontsize,
        'xtick.labelsize': tick_fontsize,
        'ytick.labelsize': tick_fontsize,
        'legend.fontsize': legend_fontsize,
        'figure.figsize': figure_size
    })
    
    # Use run_names as labels if no run_labels provided
    if run_labels is None:
        run_labels = run_names
    
    # Default line styles and colors if not provided
    if line_styles is None:
        line_styles = ['-', '--', '-.', ':'] * (len(run_names) // 4 + 1)  # Repeat if more than 4 runs
    
    if line_colors is None:
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f']
        line_colors = colors[:len(run_names)] if len(run_names) <= len(colors) else colors * (len(run_names) // len(colors) + 1)
    
    # Dictionary to store data for each run
    run_data = {}
    max_epochs = 0
    
    # Load data for each run
    for run_idx, run_name in enumerate(run_names):
        print(f"Loading data for run: {run_name}")
        
        # Find fold history files for the current run
        fold_files = [f for f in os.listdir(source_dir) if f.startswith(f"{run_name}_fold") and f.endswith("_history.csv")]
        num_folds = len(fold_files)
        
        if num_folds == 0:
            print(f"No fold history files found for run {run_name} in {source_dir}")
            continue
        
        print(f"Found {num_folds} folds for run {run_name}")
        
        # Load all fold history data
        all_fold_results = []
        
        for fold_idx in range(1, num_folds + 1):
            history_file = os.path.join(source_dir, f"{run_name}_fold{fold_idx}_history.csv")
            
            if not os.path.exists(history_file):
                print(f"Warning: History file for fold {fold_idx} not found: {history_file}")
                continue
            
            fold_df = pd.read_csv(history_file)
            
            # Structure data
            fold_results = {
                'val_accuracies': fold_df['val_acc'].values,
                'epochs': fold_df['epoch'].values if 'epoch' in fold_df.columns else np.arange(1, len(fold_df) + 1)
            }
            
            # Load ROC curve data if available
            roc_file = os.path.join(source_dir, f"{run_name}_fold{fold_idx}_val_roc.csv")
            if os.path.exists(roc_file):
                roc_df = pd.read_csv(roc_file)
                fold_results['fpr'] = roc_df['fpr'].values
                fold_results['tpr'] = roc_df['tpr'].values
            
            all_fold_results.append(fold_results)
            max_epochs = max(max_epochs, len(fold_results['val_accuracies']))
        
        # Calculate average validation accuracy across folds for this run
        run_data[run_name] = {
            'avg_val_acc': np.mean([fold['val_accuracies'] for fold in all_fold_results], axis=0),
            'num_epochs': len(all_fold_results[0]['val_accuracies']),
            'has_roc': 'fpr' in all_fold_results[0] and 'tpr' in all_fold_results[0]
        }
        
        # Calculate average ROC curve if available
        if run_data[run_name]['has_roc']:
            # Interpolate ROC curves to a common set of points for proper averaging
            # Using 100 points for interpolation
            common_fpr = np.linspace(0, 1, 100)
            interpolated_tprs = []
            
            for fold_results in all_fold_results:
                # Interpolate the TPR values at the common FPR points
                interpolated_tprs.append(np.interp(common_fpr, fold_results['fpr'], fold_results['tpr']))
                # Set the initial point to 0,0 and final point to 1,1
                interpolated_tprs[-1][0] = 0.0
                interpolated_tprs[-1][-1] = 1.0
            
            # Calculate mean TPR
            mean_tpr = np.mean(interpolated_tprs, axis=0)
            
            # Calculate standard deviation if there's more than one fold
            if len(interpolated_tprs) > 1:
                std_tpr = np.std(interpolated_tprs, axis=0)
                tpr_upper = np.minimum(mean_tpr + std_tpr, 1)
                tpr_lower = np.maximum(mean_tpr - std_tpr, 0)
                run_data[run_name]['roc_std_upper'] = tpr_upper
                run_data[run_name]['roc_std_lower'] = tpr_lower
            
            run_data[run_name]['avg_fpr'] = common_fpr
            run_data[run_name]['avg_tpr'] = mean_tpr
            
            # Calculate AUC for the average ROC curve
            from sklearn.metrics import auc
            run_data[run_name]['avg_auc'] = auc(common_fpr, mean_tpr)
    
    # Create a timestamp for the file names
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Check if we loaded data successfully for at least one run
    if not run_data:
        print("No valid run data found. Exiting.")
        return
    
    # 1. Plot average validation accuracy comparison
    plt.figure(figsize=figure_size)
    
    # Draw vertical grid lines at every 5 epochs for easier reading
    plt.grid(False, linestyle='--', alpha=0.3, axis='both')
    
    for run_idx, run_name in enumerate(run_names):
        if run_name in run_data:
            epochs = np.arange(1, run_data[run_name]['num_epochs'] + 1)
            plt.plot(epochs, run_data[run_name]['avg_val_acc'], 
                     color=line_colors[run_idx], 
                     linestyle=line_styles[run_idx],
                     linewidth=line_width,
                     label=f"{run_labels[run_idx]}")
    plt.grid(False, axis='x')  # Enable grid only for the x-axis
    plt.grid(True, axis='y')
    plt.title('Comparison of Average Validation Accuracy (5 Folds)')
    plt.xlabel('Epoch')
    plt.ylabel('Validation Accuracy')
    plt.legend()
    plt.tight_layout()
    
    # Save the comparison plot
    comparison_filename = f"validation_accuracy_comparison_{timestamp}"
    plt.savefig(f"{save_dir}/{comparison_filename}.pdf", dpi=dpi)
    plt.close()
    
    # 2. Plot ROC curves comparison if available
    roc_available = any(run_data[run_name]['has_roc'] for run_name in run_data)
    
    if roc_available:
        plt.figure(figsize=figure_size)
        
        # Plot diagonal reference line
        plt.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
        
        for run_idx, run_name in enumerate(run_names):
            if run_name in run_data and run_data[run_name]['has_roc']:
                # Plot average ROC
                plt.plot(run_data[run_name]['avg_fpr'], 
                         run_data[run_name]['avg_tpr'],
                         color=line_colors[run_idx],
                         linestyle=line_styles[run_idx],
                         linewidth=line_width,
                         label=f"{run_labels[run_idx]} (AUC = {run_data[run_name]['avg_auc']:.3f})")
                
                # Plot standard deviation ribbon if available
                if 'roc_std_upper' in run_data[run_name]:
                    plt.fill_between(run_data[run_name]['avg_fpr'],
                                    run_data[run_name]['roc_std_lower'],
                                    run_data[run_name]['roc_std_upper'],
                                    color=line_colors[run_idx],
                                    alpha=0.15)
        
        plt.title('Comparison of Average ROC Curves (5 Folds)')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.grid(False)
        plt.legend(loc='lower right')
        plt.tight_layout()
        
        # Save the ROC comparison plot
        roc_comparison_filename = f"roc_curve_comparison_{timestamp}"
        plt.savefig(f"{save_dir}/{roc_comparison_filename}.pdf", dpi=dpi)
        plt.close()
    else:
        print("ROC curve data not available for any of the runs.")
    
    print(f"Comparison plots saved to {save_dir} directory.")

# Example usage:
# compare_runs(
#     run_names=["experiment_1", "experiment_2"],
#     run_labels=["Model A", "Model B"],
#     source_dir="results",
#     save_dir="comparison_graphs",
#     line_colors=["#1f77b4", "#ff7f0e"],  # Blue and Orange
#     line_styles=["-", "--"]
# )

In [33]:
compare_runs(
    run_names=["densenet201_20250326_083352", "densenet201_20250326_123100"],
    run_labels=["DenseNet (72x72x72)", "DenseNet (36x36x36)"],
    source_dir="results",
    save_dir="comparison_graphs",
    line_colors=["#1f77b4", "#ff7f0e"],  # Blue and Orange
    line_styles=["-", "--"]
)

Loading data for run: densenet201_20250326_083352
Found 5 folds for run densenet201_20250326_083352
Loading data for run: densenet201_20250326_123100
Found 5 folds for run densenet201_20250326_123100


C:\Users\Josh\AppData\Local\Temp\ipykernel_23264\2807281524.py:131: UserWarning: First parameter to grid() is false, but line properties are supplied. The grid will be enabled.
  plt.grid(False, linestyle='--', alpha=0.3, axis='both')


Comparison plots saved to comparison_graphs directory.
